# Estud-IA · IA nivel intermedio · Clase 2
**Cómo aprende una red y cómo analizarla**

Ejecuta las celdas en orden (Ctrl + Enter). Tres partes:
1. Una imagen son números (5 × 5)
2. Usar un modelo ya entrenado: MobileNetV2 (inferencia)
3. Práctica guiada: entrenar una red pequeña con 30 mensajes y analizar sus errores

## Parte 1 · Una imagen de 5 × 5 son 25 números
0 = negro, 255 = blanco. No hay una instrucción que diga «dibuja un dos»: la figura aparece por la posición de los valores.

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

dos = tf.constant([
    [255, 255, 255, 255, 255],
    [  0,   0,   0,   0, 255],
    [255, 255, 255, 255, 255],
    [255,   0,   0,   0,   0],
    [255, 255, 255, 255, 255],
])

plt.imshow(dos, cmap="gray", vmin=0, vmax=255)
plt.show()

**Reto:** cambia uno de los valores 255 por 0 y vuelve a ejecutar. Antes de ejecutar, predice qué pasará con la figura.

In [ ]:
# MNIST: cada imagen mide 28 x 28
print("Valores de un dígito de MNIST:", 28 * 28)
print("Imagen de 10 x 10 en escala de grises:", 10 * 10, "valores")

## Parte 2 · Usar un modelo ya entrenado (inferencia)
MobileNetV2 ya aprendió 1.000 categorías con ImageNet. **No lo entrenamos: solo lo usamos.**

Antes de ejecutar: ¿lo reconocerá como gato o lo confundirá con otro animal?

In [ ]:
from skimage import data

foto = data.chelsea()          # foto de ejemplo de un gato
plt.imshow(foto)
plt.axis("off")
plt.show()
print("Forma de la foto (alto, ancho, canales):", foto.shape)

### Preparar la entrada
MobileNetV2 espera 224 × 224 píxeles con 3 canales y valores en el formato que aplica `preprocess_input`. No basta con un buen modelo: hay que darle los datos en el formato adecuado.

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input, decode_predictions

x = tf.image.resize(foto, (224, 224))
x = tf.expand_dims(x, 0)                 # un lote con una sola imagen
x = preprocess_input(tf.cast(x, tf.float32))
print("Forma de la entrada:", x.shape)

In [ ]:
modelo = MobileNetV2(weights="imagenet")   # pesos ya aprendidos
print("Parámetros del modelo:", f"{modelo.count_params():,}")

### Predecir (esto es inferencia: no se modifican los pesos)

In [ ]:
salida = modelo.predict(x)
for _, nombre, prob in decode_predictions(salida, top=3)[0]:
    print(f"{nombre:25s} {prob:.1%}")

Observa: el modelo no responde «gato» o «no gato», entrega categorías y puntuaciones. Una puntuación alta **no garantiza** que acierte.

**Pregunta:** ¿qué pasaría con una foto oscura, borrosa o un dibujo animado?

## Parte 3 · Práctica guiada: el clasificador de mensajes de soporte
Una tienda en línea quiere enviar cada mensaje al área correcta: **facturación, envío o devolución**. Entrenamos una red pequeña con 30 mensajes inventados (10 por categoría) y la probamos con 6 mensajes que nunca vio.

In [ ]:
import numpy as np
import random

SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); tf.keras.utils.set_random_seed(SEMILLA)

categorias = ["facturación", "envío", "devolución"]

entrenamiento = [
    # facturación
    ("me cobraron dos veces la misma compra", 0),
    ("la factura tiene un valor equivocado", 0),
    ("necesito mi factura electrónica", 0),
    ("me cobraron de más en el pago", 0),
    ("el cobro no coincide con el precio", 0),
    ("quiero corregir los datos de mi factura", 0),
    ("no reconozco este cargo en mi tarjeta", 0),
    ("me llegó la factura con otro nombre", 0),
    ("pagué y no aparece el recibo", 0),
    ("el total de la factura está mal", 0),
    # envío
    ("mi pedido no ha llegado", 1),
    ("dónde está mi paquete", 1),
    ("quiero cambiar la dirección de entrega", 1),
    ("el envío lleva diez días de retraso", 1),
    ("la guía de rastreo no se actualiza", 1),
    ("el mensajero no vino a entregar", 1),
    ("cuándo llega mi pedido", 1),
    ("me entregaron el paquete en otra dirección", 1),
    ("necesito saber el estado del envío", 1),
    ("el transportista dejó el pedido con un vecino", 1),
    # devolución
    ("quiero devolver el producto", 2),
    ("el producto llegó dañado y quiero cambiarlo", 2),
    ("cómo hago la devolución de mi pedido", 2),
    ("me enviaron el artículo equivocado", 2),
    ("quiero que me devuelvan el dinero", 2),
    ("el producto no funciona y lo quiero devolver", 2),
    ("necesito cambiar la talla de la camisa", 2),
    ("quiero retornar este pedido", 2),
    ("la compra no era lo que esperaba y la devuelvo", 2),
    ("solicito el reembolso por un producto defectuoso", 2),
]
textos_ent = [t for t, _ in entrenamiento]
y_ent = np.array([y for _, y in entrenamiento])
print(len(textos_ent), "mensajes de entrenamiento")

In [ ]:
# Mensajes de prueba: la red nunca los ve al entrenar
prueba = [
    ("me cobraron el envío dos veces", 0),
    ("no quiero devolver nada, solo saber dónde está mi pedido", 1),
    ("parce, nada que me llega el chunche", 1),
    ("la factura llegó pero el producto no", 1),
    ("quiero la devolución del dinero porque me cobraron de más", 0),
    ("el producto está roto", 2),
]
textos_prueba = [t for t, _ in prueba]
y_prueba = np.array([y for _, y in prueba])

### Representación: bolsa de palabras
Cada mensaje se convierte en un vector: una posición por palabra del vocabulario, con 1 si la palabra aparece. **Se pierde el orden.**

In [ ]:
vectorizador = tf.keras.layers.TextVectorization(output_mode="multi_hot")
vectorizador.adapt(textos_ent)
vocabulario = vectorizador.get_vocabulary()
print("Palabras en el vocabulario:", len(vocabulario))
print("¿«chunche» está en el vocabulario?", "chunche" in vocabulario)
print(vectorizador(["mi pedido no ha llegado"]).numpy())

### El modelo: bolsa de palabras → Dense(16, ReLU) → Dense(3, softmax)

In [ ]:
red = tf.keras.Sequential([
    vectorizador,
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(3, activation="softmax"),
])
red.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
historial = red.fit(tf.constant(textos_ent), y_ent, epochs=150, verbose=0)
red.summary()
print("Exactitud en entrenamiento:", f"{historial.history['accuracy'][-1]:.0%}")

### Antes de ver las predicciones
Escribe en el chat: **¿cuáles de los 6 mensajes crees que fallarán? ¿Por qué?**

In [ ]:
probs = red.predict(tf.constant(textos_prueba), verbose=0)
aciertos = 0
print(f"{'':2s} {'Mensaje':58s} {'Esperada':12s} {'Predicha':12s} Prob.")
for texto, esperada, p in zip(textos_prueba, y_prueba, probs):
    pred = int(np.argmax(p))
    ok = pred == esperada
    aciertos += ok
    marca = "✓" if ok else "✗"
    print(f"{marca:2s} {texto:58s} {categorias[esperada]:12s} {categorias[pred]:12s} {p[pred]:.0%}")
print(f"\nAciertos: {aciertos} de {len(textos_prueba)}")

*Las predicciones pueden variar levemente según la versión de TensorFlow y la semilla.*

## Analiza el error
Elige un acierto y un error. **Descríbelos antes de explicarlos** y ubica dónde nace cada error:

| Eslabón | Pregunta |
|---|---|
| Preparación de la entrada | ¿Qué le hicimos al dato? ¿Hay palabras desconocidas? |
| Representación | ¿Qué números ve el modelo? ¿Se perdió el orden o la negación? |
| Decisión del modelo | ¿Entre qué categorías elige y con qué ejemplos aprendió? |

Y siempre: ¿falla la etiqueta, el contexto o una limitación del modelo?

In [ ]:
# Ejemplo para explorar: ¿qué ve la red en cada mensaje?
for texto in ["no quiero devolver nada, solo saber dónde está mi pedido", "parce, nada que me llega el chunche"]:
    desconocidas = [w for w in texto.split() if w not in vocabulario]
    print(texto)
    print("  palabras que la red no conoce:", desconocidas)
    print("  probabilidades:", dict(zip(categorias, red.predict(tf.constant([texto]), verbose=0)[0].round(2))))

## Tu turno
1. Agrega 3 mensajes de entrenamiento que incluyan una negación o jerga y vuelve a entrenar. ¿Cambió el resultado?
2. Cambia `SEMILLA` y ejecuta de nuevo: ¿se mantienen los mismos errores?
3. Escribe una conclusión: **no basta con reconocer palabras; hay que comprobar la intención.**